In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.insert(0, "..")
from pathlib import Path
import numpy as np, pandas as pd

DATA = Path("../data/processed")
tix = pd.read_parquet(DATA / "tickets_weak_labels.parquet")

raw = pd.read_csv("../data/raw/twcs.csv", usecols=["tweet_id", "inbound", "created_at", "response_tweet_id"],
                  dtype={"tweet_id": "int64", "response_tweet_id": "string"})
raw["created_at"] = pd.to_datetime(raw["created_at"], format="%a %b %d %H:%M:%S %z %Y")
company_times = raw.loc[~raw["inbound"]].set_index("tweet_id")["created_at"]
resp_ids = raw.set_index("tweet_id")["response_tweet_id"]

# Option B: every linked reply id → keep the earliest COMPANY reply
ids = tix[["tweet_id"]].assign(rid=tix["tweet_id"].map(resp_ids).str.split(",")).explode("rid")
ids["rid"] = pd.to_numeric(ids["rid"], errors="coerce").astype("Int64")
ids["reply_at"] = ids["rid"].map(company_times)
first_reply = ids.groupby("tweet_id")["reply_at"].min()

tix["first_reply_at"] = tix["tweet_id"].map(first_reply)
tix["response_min"] = (tix["first_reply_at"] - tix["created_at"]).dt.total_seconds() / 60

listed_first = tix["reply_tweet_id"].astype("Int64").map(company_times)
multi = tix["tweet_id"].map(resp_ids).str.contains(",", na=False)
print(f"Tickets with multiple linked replies: {multi.mean():.1%}")
print(f"…where the earliest reply ≠ first-listed id: {(listed_first != tix['first_reply_at'])[multi].mean():.1%}")

print(f"\nMissing reply time: {tix['response_min'].isna().sum()} | non-positive: {(tix['response_min'] <= 0).sum()}")
valid = tix["response_min"] > 0
print("\nResponse time (minutes):")
print(tix.loc[valid, "response_min"].describe(percentiles=[.25, .5, .75, .9, .95, .99]).round(1))

print("\nMedian response (min) by company, top 12 by volume:")
top = tix.loc[valid, "company"].value_counts().head(12).index
print(tix[valid & tix["company"].isin(top)].groupby("company")["response_min"].median().sort_values().round(1))

tix.loc[valid].to_parquet(DATA / "ps4_base.parquet", index=False)

Tickets with multiple linked replies: 11.5%
…where the earliest reply ≠ first-listed id: 43.2%

Missing reply time: 0 | non-positive: 0

Response time (minutes):
count    100000.0
mean        280.1
std        1609.4
min           0.0
25%           6.6
50%          24.7
75%         125.7
90%         550.2
95%        1164.8
99%        4903.6
max      323473.7
Name: response_min, dtype: float64

Median response (min) by company, top 12 by volume:
company
TMobileHelp          2.6
SouthwestAir         7.4
Uber_Support         7.4
Delta               10.8
AmazonHelp          11.5
AmericanAir         11.8
Ask_Spectrum        32.3
comcastcares        42.8
SpotifyCares        64.9
AppleSupport        84.4
Tesco              107.0
British_Airways    187.7
Name: response_min, dtype: float64


#features and split

In [8]:
import joblib, time
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from src.experiments import log_experiment

MODELS = Path("../models")
df = pd.read_parquet(DATA / "ps4_base.parquet")
CAP = 7 * 24 * 60
print(f"Dropping {(df['response_min'] > CAP).sum():,} tickets answered after 7+ days "
      f"({(df['response_min'] > CAP).mean():.2%})")
df = df[df["response_min"] <= CAP].copy()

# Signals (same prep as PS-2), then predicted intent & priority from the PS-1 / PS-2 models
SIG = ["vader", "n_exclaim", "caps_ratio", "escalation_kw", "risk_kw", "calm_kw"]
df[SIG] = df[SIG].astype(float)
df["n_exclaim"] = df["n_exclaim"].clip(upper=5)
df["pred_intent"] = joblib.load(MODELS / "ps1_tfidf_lr.joblib").predict(df["text_clean"])
df["pred_priority"] = joblib.load(MODELS / "ps2_best_classical.joblib").predict(df)

# Ticket and timing features (hour/weekday in UTC; trees learn company × hour interactions)
txt = df["text"].astype(str)
df["n_chars"] = txt.str.len()
df["n_question"] = txt.str.count(r"\?")
df["has_url"] = txt.str.contains("http").astype(int)
df["hour"] = df["created_at"].dt.hour
df["dayofweek"] = df["created_at"].dt.dayofweek
df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)
top20 = df["company"].value_counts().head(20).index
df["company_grp"] = df["company"].where(df["company"].isin(top20), "Other")
df["y_log"] = np.log1p(df["response_min"])

CAT = ["company_grp", "pred_intent", "pred_priority"]
NUM = ["n_words", "n_chars", "n_question", "has_url", "hour", "dayofweek", "is_weekend"] + SIG

train, temp = train_test_split(df, test_size=0.2, random_state=42)
val, test = train_test_split(temp, test_size=0.5, random_state=42)
pre = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), CAT),
                         ("num", StandardScaler(), NUM)])
Xtr, Xva, Xte = pre.fit_transform(train), pre.transform(val), pre.transform(test)
print(f"train {len(train):,} | val {len(val):,} | test {len(test):,} | features {Xtr.shape[1]}")
print("Predicted intent mix:", df["pred_intent"].value_counts(normalize=True).round(3).to_dict())
print("Predicted priority mix:", df["pred_priority"].value_counts(normalize=True).round(3).to_dict())

Dropping 166 tickets answered after 7+ days (0.17%)
train 79,867 | val 9,983 | test 9,984 | features 42
Predicted intent mix: {'General': 0.45, 'Complaint': 0.205, 'Technical': 0.185, 'Billing': 0.116, 'Account': 0.043}
Predicted priority mix: {'Medium': 0.521, 'Low': 0.401, 'High': 0.078}


/var/folders/z_/l6wf1s_n4zbd__lspq5kqmh00000gn/T/ipykernel_28220/3332964242.py:27: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df["dayofweek"] = df["created_at"].dt.dayofweek


In [9]:
raw2 = pd.read_csv("../data/raw/twcs.csv", usecols=["author_id", "inbound", "created_at", "text"],
                   dtype={"author_id": "string", "text": "string"})
raw2["created_at"] = pd.to_datetime(raw2["created_at"], format="%a %b %d %H:%M:%S %z %Y")
to_ns = lambda s: s.dt.tz_localize(None).values.astype("datetime64[ns]").astype("int64")

companies = {c.lower(): c for c in df["company"].unique()}
replies = raw2[~raw2["inbound"] & raw2["author_id"].isin(companies.values())]
requests = raw2[raw2["inbound"]].copy()
requests["company"] = requests["text"].str.extract(r"@(\w+)", expand=False).str.lower().map(companies)
requests = requests.dropna(subset=["company"])

reply_times = {c: np.sort(to_ns(g["created_at"])) for c, g in replies.groupby("author_id")}
request_times = {c: np.sort(to_ns(g["created_at"])) for c, g in requests.groupby("company")}

def window_count(times_by_co, tickets, minutes):
    """For each ticket, count the company's events in [t - window, t): past only, no leakage."""
    res = np.zeros(len(tickets), dtype="int64")
    t_ns, w = to_ns(tickets["created_at"]), int(minutes * 60 * 1e9)
    for co, pos in tickets.groupby("company").indices.items():
        ev = times_by_co.get(co)
        if ev is not None:
            res[pos] = np.searchsorted(ev, t_ns[pos]) - np.searchsorted(ev, t_ns[pos] - w)
    return res

df["replies_prev_60m"] = window_count(reply_times, df, 60)
df["requests_prev_60m"] = window_count(request_times, df, 60)
print(df[["replies_prev_60m", "requests_prev_60m"]].describe().round(1))
print("\nSpearman correlation with log response time:")
print(df[["replies_prev_60m", "requests_prev_60m", "y_log"]].corr(method="spearman")["y_log"].round(3))

# Rebuild the split (same seed → same rows) with the new features
NUM = list(dict.fromkeys(NUM + ["replies_prev_60m", "requests_prev_60m"]))  # no duplicates on rerun
train, temp = train_test_split(df, test_size=0.2, random_state=42)
val, test = train_test_split(temp, test_size=0.5, random_state=42)
pre = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), CAT), ("num", StandardScaler(), NUM)])
Xtr, Xva, Xte = pre.fit_transform(train), pre.transform(val), pre.transform(test)
print(f"\nfeatures now {Xtr.shape[1]}")

       replies_prev_60m  requests_prev_60m
count           99834.0            99834.0
mean               40.3               32.6
std                51.8               36.0
min                 0.0                0.0
25%                 9.0                8.0
50%                23.0               21.0
75%                46.0               41.0
max               494.0              319.0

Spearman correlation with log response time:
replies_prev_60m    -0.216
requests_prev_60m   -0.129
y_log                1.000
Name: y_log, dtype: float64

features now 44


#baseline and three models

In [10]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error, r2_score
from xgboost import XGBRegressor

def reg_scores(y_log_true, y_log_pred):
    y_log_true, y_log_pred = np.asarray(y_log_true, dtype=float), np.asarray(y_log_pred, dtype=float)
    yt, yp = np.expm1(y_log_true), np.clip(np.expm1(y_log_pred), 0, None)
    return {"MAE_min": mean_absolute_error(yt, yp), "RMSE_min": mean_squared_error(yt, yp) ** 0.5,
            "MedAE_min": median_absolute_error(yt, yp),
            "R2_log": r2_score(y_log_true, y_log_pred), "R2_min": r2_score(yt, yp)}

def show(name, m, secs=0):
    print(f"{name:22} MAE {m['MAE_min']:7.1f} | RMSE {m['RMSE_min']:7.1f} | MedAE {m['MedAE_min']:6.1f} "
          f"| R² log {m['R2_log']:.3f} | R² min {m['R2_min']:.3f} | {secs:.0f}s")

# Naive baseline: each company's median log-time from train
co_med = train.groupby("company_grp")["y_log"].median()
m_naive = reg_scores(val["y_log"], val["company_grp"].map(co_med).fillna(train["y_log"].median()))
log_experiment("PS-4", "Naive company median", {"eval": "val"}, m_naive)
show("Naive company median", m_naive)

models = {
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6, subsample=0.8,
                            colsample_bytree=0.8, early_stopping_rounds=50, eval_metric="rmse",
                            n_jobs=-1, random_state=42),
}
results = {}
for name, mdl in models.items():
    t0 = time.time()
    if name == "XGBoost":
        mdl.fit(Xtr, train["y_log"], eval_set=[(Xva, val["y_log"])], verbose=False)
    else:
        mdl.fit(Xtr, train["y_log"])
    m = reg_scores(val["y_log"], mdl.predict(Xva))
    results[name] = (mdl, m)
    log_experiment("PS-4", name, {"target": "log1p(minutes)", "cap_days": 7, "eval": "val"}, m)
    show(name, m, time.time() - t0)

Naive company median   MAE   215.9 | RMSE   775.1 | MedAE   29.2 | R² log 0.220 | R² min -0.019 | 0s
LinearRegression       MAE   215.3 | RMSE   777.1 | MedAE   34.9 | R² log 0.253 | R² min -0.024 | 0s
RandomForest           MAE   192.9 | RMSE   747.0 | MedAE   21.1 | R² log 0.413 | R² min 0.053 | 14s
XGBoost                MAE   194.8 | RMSE   747.6 | MedAE   22.1 | R² log 0.413 | R² min 0.052 | 2s


In [11]:
from scipy.stats import randint, uniform, loguniform
from sklearn.model_selection import RandomizedSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance
from sklearn.metrics import precision_score, recall_score, f1_score

search = RandomizedSearchCV(
    XGBRegressor(eval_metric="rmse", n_jobs=-1, random_state=42),
    {"n_estimators": randint(200, 900), "learning_rate": loguniform(0.02, 0.2),
     "max_depth": randint(3, 10), "min_child_weight": randint(1, 20),
     "subsample": uniform(0.6, 0.4), "colsample_bytree": uniform(0.5, 0.5),
     "reg_lambda": loguniform(0.1, 10)},
    n_iter=30, cv=KFold(5, shuffle=True, random_state=42),
    scoring="neg_root_mean_squared_error", random_state=42, n_jobs=1, verbose=1)
t0 = time.time()
search.fit(Xtr, train["y_log"])
print(f"CV done in {(time.time() - t0) / 60:.1f} min | best CV RMSE (log) {-search.best_score_:.4f}")
print("Best params:", {k: round(v, 3) if isinstance(v, float) else v for k, v in search.best_params_.items()})

best = search.best_estimator_
show("XGBoost tuned (val)", reg_scores(val["y_log"], best.predict(Xva)))
m_test = reg_scores(test["y_log"], best.predict(Xte))
show("XGBoost tuned (TEST)", m_test)
log_experiment("PS-4", "XGBoost tuned (5-fold CV)", {**search.best_params_, "eval": "test"}, m_test)

# SLA early warning: flag tickets predicted to take > 60 min (threshold tuned on val by F1)
SLA = 60
pv, bv = best.predict(Xva), val["response_min"] > SLA
ths = np.log1p(np.arange(10, 121, 5))
best_th = max(ths, key=lambda th: f1_score(bv, pv > th))
pt, bt = best.predict(Xte) > best_th, test["response_min"] > SLA
print(f"\nSLA > {SLA} min | base rate {bt.mean():.1%} | flag if predicted > {np.expm1(best_th):.0f} min "
      f"| precision {precision_score(bt, pt):.3f} | recall {recall_score(bt, pt):.3f} | F1 {f1_score(bt, pt):.3f}")

# Permutation importance on the original columns
pipe = Pipeline([("pre", pre), ("xgb", best)])
cols = CAT + NUM
pi = permutation_importance(pipe, val[cols], val["y_log"], scoring="r2", n_repeats=5, random_state=42, n_jobs=1)
imp = pd.Series(pi.importances_mean, index=cols).sort_values(ascending=False)
print("\nPermutation importance (drop in R² when shuffled):\n", imp.round(4).head(12))
imp.to_csv("../reports/ps4_feature_importance.csv")

joblib.dump({"pipeline": pipe, "cat": CAT, "num": NUM, "cap_min": CAP, "sla_min": SLA,
             "sla_threshold_log": best_th}, MODELS / "ps4_xgb_pipeline.joblib")

Fitting 5 folds for each of 30 candidates, totalling 150 fits
CV done in 3.1 min | best CV RMSE (log) 1.4360
Best params: {'colsample_bytree': np.float64(0.961), 'learning_rate': np.float64(0.025), 'max_depth': 9, 'min_child_weight': 8, 'n_estimators': 537, 'reg_lambda': np.float64(1.201), 'subsample': np.float64(0.835)}
XGBoost tuned (val)    MAE   193.0 | RMSE   742.2 | MedAE   21.5 | R² log 0.418 | R² min 0.066 | 0s
XGBoost tuned (TEST)   MAE   213.0 | RMSE   816.8 | MedAE   21.0 | R² log 0.411 | R² min 0.064 | 0s

SLA > 60 min | base rate 34.1% | flag if predicted > 35 min | precision 0.625 | recall 0.777 | F1 0.693

Permutation importance (drop in R² when shuffled):
 company_grp          0.4632
requests_prev_60m    0.4611
replies_prev_60m     0.4124
hour                 0.0689
dayofweek            0.0344
n_chars              0.0204
n_words              0.0142
pred_intent          0.0123
vader                0.0036
has_url              0.0025
caps_ratio           0.0022
escalation_

['../models/ps4_xgb_pipeline.joblib']

In [12]:
%pip install ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.3/217.3 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 16.3 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
